# Auliya Robusta SNI — reproduksi adaptif fold A

YOLO26s-seg, 21 kelas, seed 0. Pilih satu arm: RGB, AF2, atau HAAR. Split resmi A: 676 train, 119 validasi; grup A tidak diekstrak.

Paket data pilot lama tetap dipakai. Taruh paket kode `auliya-official-frequency-source-v1.zip` di `Coffee_Bean_Detection/bundles/` di samping paket pilot. Secara default notebook menjalankan **preflight tanpa training**. Setelah preflight lulus dan GPU siap, set `AUTHORIZE_TRAINING = True` di sel terakhir. Pakai maksimal satu sesi per arm.


In [ ]:
from google.colab import drive
from pathlib import Path, PurePosixPath
import hashlib, json, os, shutil, subprocess, sys, time
from urllib.request import urlopen
from zipfile import ZipFile

ARM = 'RGB'  # RGB, AF2, atau HAAR
assert ARM in {'RGB', 'AF2', 'HAAR'}
drive.mount('/content/drive')
drive_root = Path('/content/drive')
pilot_name = 'auliya-robusta-sni-frequency-colab-v1.zip'
code_name = 'auliya-official-frequency-source-v1.zip'
code_sha = '8db0764e3e2148ea1e0cea85454bdb58110bc0962aee0e9bd333e1c946a1eaa0'
canonical = drive_root / 'MyDrive/Coffee_Bean_Detection'
shortcuts = drive_root / '.shortcut-targets-by-id'
shared = sorted(shortcuts.glob('*/Coffee_Bean_Detection'))
candidates = [p for p in (canonical, *shared) if (p/'bundles'/pilot_name).is_file() and (p/'bundles'/code_name).is_file()]
if canonical in candidates:
    PROJECT = canonical
elif len(candidates) == 1:
    PROJECT = candidates[0]
else:
    raise RuntimeError(f'Butuh satu folder proyek dengan kedua ZIP di bundles/: {candidates}. Cek shortcut Drive dan paket kode.')
PILOT = PROJECT/'bundles'/pilot_name
CODE = PROJECT/'bundles'/code_name
WORK = Path('/content') / f'auliya-official-{code_sha[:12]}'
WORK.mkdir(exist_ok=True)

def digest(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def fetch_verified(url, destination, sha):
    if destination.exists():
        if digest(destination) != sha:
            raise RuntimeError(f'SHA256 tidak cocok: {destination}')
        return destination
    temporary = destination.with_suffix(destination.suffix + '.part')
    with urlopen(url, timeout=60) as remote, temporary.open('wb') as local:
        shutil.copyfileobj(remote, local, length=4*1024*1024)
    if digest(temporary) != sha:
        temporary.unlink()
        raise RuntimeError(f'SHA256 unduhan tidak cocok: {destination.name}')
    temporary.replace(destination)
    return destination

if digest(CODE) != code_sha:
    raise RuntimeError('Paket kode berbeda dari versi notebook ini.')
with ZipFile(CODE) as source:
    for entry in source.namelist():
        p = PurePosixPath(entry)
        if p.is_absolute() or '..' in p.parts or p.parts[:2] != ('src', 'coffee_detector') or p.suffix != '.py':
            raise RuntimeError(f'Entri kode tidak aman: {entry}')
    source.extractall(WORK)
ENV = {**os.environ, 'PYTHONPATH': str(WORK/'src'), 'PYTHONUNBUFFERED': '1', 'OMP_NUM_THREADS': '2'}
def run(module, *args):
    subprocess.run([sys.executable, '-u', '-m', module, *map(str, args)], cwd=WORK, env=ENV, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-q', 'ultralytics==8.4.210'], check=True)
print('PROYEK:', PROJECT, '| ARM:', ARM)


In [ ]:
metadata = fetch_verified(
    'https://zenodo.org/api/records/24346584/files/splits_metadata_results_scripts.zip/content',
    WORK/'zenodo_metadata.zip',
    '798bb53b0cfad35dfcfbcc463fbcb79ff4af61f9807803d6bcc65d37f7c8feaa',
)
weights = fetch_verified(
    'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26s-seg.pt',
    WORK/'yolo26s-seg.pt',
    '3da1d83e31caec96f9300eb4064f4f62882c133c7c264d63dfe61a7c197837a4',
)
DATA = WORK/'fold_a'
if not DATA.exists():
    run('coffee_detector.experiments.prepare_auliya_official_fold',
        '--metadata', metadata, '--pilot-bundle', PILOT, '--output', DATA)
RUNNER = 'coffee_detector.experiments.run_auliya_official_frequency'
args = ('--development', DATA, '--weights', weights)
run(RUNNER, *args)  # no training, no test


In [ ]:
AUTHORIZE_TRAINING = False  # Set True only after the preflight passes.

if not AUTHORIZE_TRAINING:
    print('Training tidak dijalankan. Ubah AUTHORIZE_TRAINING jika sudah siap.')
else:
    subprocess.run([sys.executable, '-c',
                    "import torch; assert torch.cuda.is_available(), 'Aktifkan GPU Colab'; print(torch.cuda.get_device_name(0))"],
                   cwd=WORK, env=ENV, check=True)
    output = PROJECT/'experiments/auliya-official-fold-a-frequency-v1'
    arm_dir = output/ARM
    arm_dir.mkdir(parents=True, exist_ok=True)
    log = arm_dir/'training.log'
    marker = arm_dir/'progress.json'
    command = [sys.executable, '-u', '-m', RUNNER, *map(str, args),
               '--arm', ARM, '--output', str(output), '--device', '0', '--authorize-training']
    print('START/RESUME:', ARM, '| LOG:', log, flush=True)
    previous = None
    with log.open('a', encoding='utf-8') as stream:
        process = subprocess.Popen(command, cwd=WORK, env=ENV, stdout=stream, stderr=subprocess.STDOUT)
        while process.poll() is None:
            if marker.is_file():
                state = json.loads(marker.read_text(encoding='utf-8'))
                if state['epoch'] != previous:
                    previous = state['epoch']
                    print(f"{ARM}: {previous}/{state['maximum_epochs']} epoch tersimpan", flush=True)
            time.sleep(15)
    if process.returncode:
        print('\n'.join(log.read_text(encoding='utf-8', errors='replace').splitlines()[-40:]))
        raise RuntimeError(f'{ARM} gagal; log={log}')
    print((arm_dir/'validation_result.json').read_text(encoding='utf-8'))
